# 🚀 Stack 2.9 — 7B QLoRA Fine-tune on Kaggle (Smart 20K)

**Base:** Qwen2.5-Coder-7B-Instruct
**Data:** my-ai-stack/stack-2-9-tool-20k-examples (20K smart examples)
**Output:** my-ai-stack/Stack-2.9-7B-finetuned
**Runtime:** GPU T4 16GB | **Time:** ~6-8 hours | **Cost:** FREE

## ⚠️ Before Starting

1. Go to **Add-ons → Secrets** and add:
   - `HF_TOKEN` = your HuggingFace write token (starts with `hf_`)
2. Set **Accelerator** to **GPU T4**
3. Internet must be ON

## Step 1: Clone & Install

In [ ]:
import os
os.chdir("/kaggle/working")

# Clone repo
!git clone https://github.com/my-ai-stack/stack-2.9.git

# Install
!pip install -q transformers>=4.40.0 peft datasets bitsandbytes accelerate huggingface_hub flash-attn --no-build-isolation

# Verify
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Step 2: Login to HuggingFace

In [ ]:
from huggingface_hub import login
import os

# Read token from Kaggle secret
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token)
print("✅ Logged into HuggingFace")

## Step 3: Download Smart 20K Dataset

In [ ]:
from huggingface_hub import hf_hub_download
import os

DATA_DIR = "/kaggle/working/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Download smart 20K dataset from org
print("Downloading smart 20K dataset...")
path = hf_hub_download(
    repo_id="my-ai-stack/stack-2-9-tool-20k-examples",
    filename="tool_examples_smart_20k.jsonl",
    repo_type="dataset",
    local_dir=DATA_DIR,
)
print(f"Dataset: {path}")

# Count
with open(path) as f:
    n_lines = sum(1 for _ in f)
print(f"Examples: {n_lines:,}")

## Step 4: Setup Data Pipeline

In [ ]:
import json
import torch
from transformers import AutoTokenizer
from datasets import load_dataset

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_LENGTH = 2048  # T4 fits this with QLoRA

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

print("Loading dataset...")
raw = load_dataset("json", data_files=path, split="train")
print(f"Loaded {len(raw)} examples")

## Step 5: Tokenize Data (messages format)

In [ ]:
def format_conversation(example):
    messages = example["messages"]
    text = ""
    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "") or ""
        tc = msg.get("tool_calls", [])
        if role == "system":
            text += f"<|im_start|>system\n{content}<|im_end|>\n"
        elif role == "user":
            text += f"<|im_start|>user\n{content}<|im_end|>\n"
        elif role == "assistant":
            if tc:
                for t in tc:
                    fn = t["function"]
                    text += f"<|im_start|>assistant\n<tool_call>\n<name>{fn['name']}</name>\n<args>\n{fn['arguments']}\n</args>\n</tool_call>\n"
                if content:
                    text += f"{content}<|im_end|>\n"
                else:
                    text += "<|im_end|>\n"
            else:
                text += f"<|im_start|>assistant\n{content}<|im_end|>\n"
        elif role == "tool":
            text += f"<|im_start|>tool\n{content}<|im_end|>\n"
    text += "<|im_start|>assistant\n"
    return {"text": text}

print("Tokenizing...")
formatted = raw.map(format_conversation)

def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    # Mask user/system/tool tokens
    input_ids = tokens["input_ids"]
    labels = tokens["labels"]
    
    # Find last assistant start
    asr = tokenizer.encode("<|im_start|>assistant", add_special_tokens=False)
    found = -1
    for i in range(len(input_ids) - len(asr) + 1):
        if input_ids[i:i+len(asr)] == asr:
            found = i
    
    if found >= 0:
        for j in range(found + len(asr)):
            labels[j] = -100
    
    # Mask padding
    for j, m in enumerate(tokens["attention_mask"]):
        if m == 0:
            labels[j] = -100
    
    return tokens

tokenized = formatted.map(tokenize, remove_columns=formatted.column_names, desc="Tokenizing")
tokenized = tokenized.filter(lambda x: x["labels"] is not None)

# Split
split = tokenized.train_test_split(test_size=0.05)
train_ds = split["train"]
val_ds = split["test"]
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

## Step 6: Load Model + LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import BitsAndBytesConfig
import torch

# Quantization config for T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print("Loading model...")
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False

# LoRA config
lora_cfg = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## Step 7: Train

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollator
import os

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=32,  # Effective batch = 32
        num_train_epochs=3,
        learning_rate=1e-4,
        fp16=True,
        bf16=False,
        warmup_ratio=0.05,
        max_grad_norm=0.3,
        logging_steps=10,
        save_steps=100,
        eval_steps=100,
        save_total_limit=2,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        remove_unused_columns=False,
        report_to="none",
    ),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

print("Starting training...")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

model.train()
trainer.train()
print("✅ Training complete!")

## Step 8: Save & Upload

In [ ]:
print("Saving adapter...")
adapter_path = "/kaggle/working/final_adapter"
model.save_pretrained(adapter_path)

print("Merging with base model...")
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, adapter_path).merge_and_unload()

merged_path = "/kaggle/working/merged_model"
merged.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)

print(f"Model saved to {merged_path}")

print("Uploading to HuggingFace...")
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path=merged_path,
    repo_id="my-ai-stack/Stack-2.9-7B-finetuned",
    repo_type="model",
)
print("🎉 Done! Model uploaded to my-ai-stack/Stack-2.9-7B-finetuned")